In [4]:
pip install pyxlsb

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import shutil
from pathlib import Path
import pandas as pd
import tensorflow as tf
import kagglehub
from sklearn.preprocessing import LabelEncoder
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

In [2]:
attributes_df = pd.read_excel("list_attr_celeba_with_identity.xlsb", engine='pyxlsb')

In [4]:
print(f"Original dataset size: {len(attributes_df)}")
attributes_df = attributes_df.dropna(subset=['image_id', 'Celeb_identity'])
print(f"After removing missing values: {len(attributes_df)}")


Original dataset size: 1048575
After removing missing values: 202599


In [5]:
attributes_df['Celeb_identity'] = attributes_df['Celeb_identity'].astype(int)

In [3]:
train_dir = kagglehub.dataset_download("jessicali9530/celeba-dataset")  # Download dataset
image_dir = os.path.join(train_dir, "img_align_celeba", "img_align_celeba")  # Path to images

100%|█████████████████████████████████████████████████████████████████████████████| 1.33G/1.33G [03:41<00:00, 6.46MB/s]

Extracting files...


In [6]:
image_paths = [os.path.join(image_dir, img_name) for img_name in attributes_df['image_id']]

# Encode identity labels to integers (0 to num_classes-1)
label_encoder = LabelEncoder()
identity_labels = label_encoder.fit_transform(attributes_df['Celeb_identity'])
num_classes = len(label_encoder.classes_)

print(f"Total images: {len(image_paths)}")
print(f"Number of classes (identities): {num_classes}")

Total images: 202599
Number of classes (identities): 10177


In [7]:
def create_celebrity_folder(celebrity_id, attributes_df, image_dir, output_base_dir='celebrity_folders'):
    
    # Filter images for celebrity
    celebrity_images = attributes_df[attributes_df['Celeb_identity'] == celebrity_id]['image_id'].tolist()
    
    if len(celebrity_images) == 0:
        print(f"No images found for Celebrity ID {celebrity_id}")
        return None, 0
    
    # Create output directory
    celebrity_folder = os.path.join(output_base_dir, f'celebrity_{celebrity_id}')
    os.makedirs(celebrity_folder, exist_ok=True)
    
    print(f"Creating folder: {celebrity_folder}")
    print(f"Found {len(celebrity_images)} images for Celebrity ID {celebrity_id}")
    
    # Copy images
    copied_count = 0
    failed_count = 0
    
    for img_name in celebrity_images:
        source_path = os.path.join(image_dir, img_name)
        dest_path = os.path.join(celebrity_folder, img_name)
        
        try:
            if os.path.exists(source_path):
                shutil.copy2(source_path, dest_path)
                copied_count += 1
            else:
                print(f"Warning: Image not found: {source_path}")
                failed_count += 1
        except Exception as e:
            print(f"Error copying {img_name}: {e}")
            failed_count += 1
    
    print(f"\nCompleted!")
    print(f"Successfully copied: {copied_count} images")
    if failed_count > 0:
        print(f"Failed to copy: {failed_count} images")
    print(f"Folder location: {os.path.abspath(celebrity_folder)}")
    
    return celebrity_folder, copied_count


In [8]:
folder_path, num_images = create_celebrity_folder(
    celebrity_id=3699,
    attributes_df=attributes_df,
    image_dir=image_dir,
    output_base_dir='celebrity_folders'
)

Creating folder: celebrity_folders\celebrity_3699
Found 34 images for Celebrity ID 3699

Completed!
Successfully copied: 34 images
Folder location: C:\Users\deepi\Deep Learning with AI\Discriminative Deep Learning Project\celebrity_folders\celebrity_3699


In [12]:
def create_augmentation_pipeline():
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),  # 10% rotation
        tf.keras.layers.RandomZoom(0.1),  # 10% zoom
        tf.keras.layers.RandomBrightness(factor=(-0.15, 0.15)),  # Brightness adjustment
        tf.keras.layers.RandomContrast(0.2),  # Contrast adjustment
    ])
    return data_augmentation

In [13]:
def augment_celebrity_folder(celebrity_folder, target_count=100, output_folder=None):
    
    # Setup output folder
    if output_folder is None:
        output_folder = os.path.join(celebrity_folder, 'augmented')
    os.makedirs(output_folder, exist_ok=True)
    
    # Get existing images
    image_files = [f for f in os.listdir(celebrity_folder) 
                   if f.lower().endswith(('.jpg', '.jpeg', '.png')) 
                   and os.path.isfile(os.path.join(celebrity_folder, f))]
    
    current_count = len(image_files)
    
    print(f"Celebrity Folder: {celebrity_folder}")
    print(f"Current images: {current_count}")
    print(f"Target images: {target_count}")
    
    if current_count >= target_count:
        print(f"Already have {current_count} images. No augmentation needed.")
        return output_folder, 0
    
    needed_count = target_count - current_count
    print(f"Need to create: {needed_count} augmented images")
    print("="*70)
    
    # Copy original images to output folder
    print("Step 1: Copying original images...")
    for img_file in image_files:
        source = os.path.join(celebrity_folder, img_file)
        dest = os.path.join(output_folder, img_file)
        if not os.path.exists(dest):
            Image.open(source).save(dest)
    print(f"Copied {current_count} original images")
    
    # Create augmentation pipeline
    augmentor = create_augmentation_pipeline()
    
    # Generate augmented images
    print("\nStep 2: Generating augmented images...")
    augmented_count = 0
    
    # Calculate how many augmentations per image
    augmentations_per_image = int(np.ceil(needed_count / current_count))
    
    for img_idx, img_file in enumerate(image_files):
        # Load image
        img_path = os.path.join(celebrity_folder, img_file)
        img = tf.io.read_file(img_path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [128, 128])
        img = tf.cast(img, tf.float32) / 255.0
        
        # Generate multiple augmentations from this image
        for aug_idx in range(augmentations_per_image):
            if augmented_count >= needed_count:
                break
            
            # Apply augmentation
            img_batch = tf.expand_dims(img, 0)
            augmented_img = augmentor(img_batch, training=True)
            augmented_img = tf.squeeze(augmented_img, 0)
            
            # Convert back to [0, 255] and save
            augmented_img = (augmented_img.numpy() * 255).astype(np.uint8)
            augmented_img_pil = Image.fromarray(augmented_img)
            
            # Create filename
            base_name = os.path.splitext(img_file)[0]
            aug_filename = f"{base_name}_aug_{aug_idx}.jpg"
            aug_path = os.path.join(output_folder, aug_filename)
            
            augmented_img_pil.save(aug_path, quality=95)
            augmented_count += 1
            
            if augmented_count % 10 == 0:
                print(f"Created {augmented_count}/{needed_count} augmented images...")
        
        if augmented_count >= needed_count:
            break
    
    final_count = current_count + augmented_count
    
    print("\n" + "="*70)
    print("AUGMENTATION COMPLETE")
    print("="*70)
    print(f"Original images: {current_count}")
    print(f"Augmented images created: {augmented_count}")
    print(f"Total images now: {final_count}")
    print(f"Output folder: {os.path.abspath(output_folder)}")
    
    return output_folder, augmented_count

In [14]:
augment_celebrity_folder("celebrity_folders/celebrity_3699", target_count=100, output_folder=None)

Celebrity Folder: celebrity_folders/celebrity_3699
Current images: 34
Target images: 100
Need to create: 66 augmented images
Step 1: Copying original images...
Copied 34 original images

Step 2: Generating augmented images...
Created 10/66 augmented images...
Created 20/66 augmented images...
Created 30/66 augmented images...
Created 40/66 augmented images...
Created 50/66 augmented images...
Created 60/66 augmented images...

AUGMENTATION COMPLETE
Original images: 34
Augmented images created: 66
Total images now: 100
Output folder: C:\Users\deepi\Deep Learning with AI\Discriminative Deep Learning Project\celebrity_folders\celebrity_3699\augmented


('celebrity_folders/celebrity_3699\\augmented', 66)